# Conditional kNN vs Concat Samples

This notebook estimates a local empirical conditional distribution around one track condition.

Protocol:

1. Choose a reference example `x_ref` and a track mask `mask`.
2. Build the condition `y = x_ref * mask`.
3. Find `K_NEIGHBORS` train examples whose masked observations are closest to `y`.
4. Visualize the neighbors, differences to `x_ref`, and MSE metrics.
5. Generate `N_MODEL_SAMPLES` from the concat model under the same condition.
6. Cache generated samples under `DATA_ROOT/conditional_knn_concat_samples` so reruns do not resample.
7. Compare local real neighbors and generated samples with permutation energy/MMD tests on domain features.

Important assumption: the kNN set is only an approximation of `p_data(x | y, mask)`. It assumes conditional smoothness in the selected masked-observation metric.


In [ ]:
import glob
import json
import os
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import stats
from tqdm.auto import tqdm

from fid.export_concat_samples import load_concat_sampler
from utils import (
    NpyImageDataset,
    channel_denormalize,
    channel_normalize,
    generate_satellite_track_mask,
    get_device,
    make_difference_plot,
    make_plot,
    make_scalar_plot_grid,
)

# Server defaults. Override with env vars only if the server layout changes.
REPO_DIR = os.environ.get('REPO_DIR', '/home')
DATA_ROOT = os.environ.get('DATA_ROOT', '/mnt/sciml/a.sadreev/sea_ice_data')

os.chdir(REPO_DIR)

IMAGE_SIZE = (320, 256)
REFERENCE_SPLIT = 'valid'
NEIGHBOR_SPLIT = 'train'
REFERENCE_INDEX = 0
EXCLUDE_REFERENCE_FROM_NEIGHBORS = True

# Track condition. If TRACK_MASK_PATH is None, the mask is generated reproducibly.
TRACK_MASK_PATH = None
TRACK_SEED = 20260407
N_TRACKS_RANGE = (7, 7)

K_NEIGHBORS = 32
N_MODEL_SAMPLES = 64
GENERATION_BATCH_SIZE = 4
GENERATION_SEED = 1234
NUM_TIMESTEPS = 50
METHOD = 'euler'

CHECKPOINT_DIR = os.path.join(REPO_DIR, 'checkpoints')
CHECKPOINT_NAME = 'ema_best_model.pth'
CONCAT_RUN_DIR = None

# kNN distance is evaluated in normalized training units on y = x * mask.
KNN_CHANNEL_WEIGHTS = np.array([1.0, 1.0], dtype=np.float32)

# Statistical tests use lower-dimensional features instead of raw pixels.
VALID_PIXEL_SAMPLE_SIZE = 20_000
N_RANDOM_PROJECTIONS = 64
TEST_N_PERMUTATIONS = 999
TEST_SEED = 4321
FEATURE_CHANNELS = (0, 1)

CACHE_ROOT = os.path.join(DATA_ROOT, 'conditional_knn_concat_samples')
CACHE_TAG = (
    f'{REFERENCE_SPLIT}_idx{REFERENCE_INDEX}_tracks{N_TRACKS_RANGE[0]}-{N_TRACKS_RANGE[1]}'
    f'_trackseed{TRACK_SEED}_n{N_MODEL_SAMPLES}_steps{NUM_TIMESTEPS}_{METHOD}'
)
CACHE_DIR = os.path.join(CACHE_ROOT, CACHE_TAG)

DEVICE = get_device()
STATS_JSON = os.path.join(DATA_ROOT, 'train', 'stats.json')
MASK_PATH = os.path.join(DATA_ROOT, 'mask_padding.npy')

with open(STATS_JSON) as f:
    stats_payload = json.load(f)
CHANNEL_MEAN = tuple(stats_payload['mean'])
CHANNEL_STD = tuple(stats_payload['std'])
VALID_MASK = np.load(MASK_PATH).astype(np.float32)

print('repo_dir            =', REPO_DIR)
print('data_root           =', DATA_ROOT)
print('device              =', DEVICE)
print('reference_split     =', REFERENCE_SPLIT)
print('neighbor_split      =', NEIGHBOR_SPLIT)
print('reference_index     =', REFERENCE_INDEX)
print('exclude_reference   =', EXCLUDE_REFERENCE_FROM_NEIGHBORS)
print('k_neighbors         =', K_NEIGHBORS)
print('n_model_samples     =', N_MODEL_SAMPLES)
print('num_timesteps       =', NUM_TIMESTEPS)
print('method              =', METHOD)
print('cache_dir           =', CACHE_DIR)
print('channel_mean        =', CHANNEL_MEAN)
print('channel_std         =', CHANNEL_STD)


In [ ]:
def resolve_concat_run_dir():
    if CONCAT_RUN_DIR is not None:
        return CONCAT_RUN_DIR
    candidates = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, '**', CHECKPOINT_NAME), recursive=True))
    assert candidates, f'No {CHECKPOINT_NAME} found under {CHECKPOINT_DIR}'
    return os.path.dirname(candidates[-1])


def to_channel_first(arr):
    arr = np.asarray(arr)
    if arr.ndim == 2:
        return arr[None, ...].astype(np.float32, copy=False)
    if arr.ndim != 3:
        raise ValueError(f'Unsupported array shape {arr.shape}')
    if arr.shape[0] in (1, 2, 3):
        out = arr
    elif arr.shape[-1] in (1, 2, 3):
        out = np.moveaxis(arr, -1, 0)
    else:
        raise ValueError(f'Unsupported array shape {arr.shape}')
    return out.astype(np.float32, copy=False)


def normalize_np(arr):
    tensor = torch.from_numpy(to_channel_first(arr).copy())
    return channel_normalize(tensor, CHANNEL_MEAN, CHANNEL_STD)


def denormalize_tensor(tensor):
    return channel_denormalize(tensor.detach().cpu(), CHANNEL_MEAN, CHANNEL_STD)


def load_denorm_stack(folder, file_names):
    arrays = [to_channel_first(np.load(os.path.join(folder, name), mmap_mode='r')) for name in file_names]
    return np.stack(arrays, axis=0).astype(np.float32, copy=False)


def make_track_mask():
    if TRACK_MASK_PATH is not None:
        mask = np.load(TRACK_MASK_PATH).astype(np.float32)
        if mask.ndim == 3:
            mask = mask[0]
        assert mask.shape == IMAGE_SIZE, f'Expected mask shape {IMAGE_SIZE}, got {mask.shape}'
        return mask * VALID_MASK

    rng_state = np.random.get_state()
    np.random.seed(TRACK_SEED)
    try:
        mask = generate_satellite_track_mask(
            image_size=IMAGE_SIZE,
            batch_size=1,
            valid_mask=VALID_MASK,
            n_tracks_range=N_TRACKS_RANGE,
        )[0]
    finally:
        np.random.set_state(rng_state)
    return mask.astype(np.float32, copy=False)


def masked_condition_mse(candidate_norm, observed_norm, mask_2d):
    mask = torch.as_tensor(mask_2d, dtype=candidate_norm.dtype).view(1, *mask_2d.shape)
    weights = torch.as_tensor(KNN_CHANNEL_WEIGHTS, dtype=candidate_norm.dtype).view(-1, 1, 1)
    diff2 = torch.square((candidate_norm * mask - observed_norm) * weights)
    denom = (mask.sum() * weights.numel()).clamp(min=1.0)
    return float(diff2.sum().item() / denom.item())


def physical_mse_rows(samples_norm, reference_norm, valid_mask):
    samples = denormalize_tensor(samples_norm).numpy()
    ref = denormalize_tensor(reference_norm.unsqueeze(0)).numpy()[0]
    mask = valid_mask.astype(bool)
    rows = []
    for idx, sample in enumerate(samples):
        row = {'idx': idx}
        row['mse_all_channels'] = float(np.mean((sample[:, mask] - ref[:, mask]) ** 2))
        for ch in range(sample.shape[0]):
            row[f'mse_ch{ch}'] = float(np.mean((sample[ch, mask] - ref[ch, mask]) ** 2))
        rows.append(row)
    return rows


def print_rows(rows, max_rows=40):
    if not rows:
        print('(empty)')
        return
    keys = list(rows[0].keys())
    print(' | '.join(keys))
    print(' | '.join(['---'] * len(keys)))
    for row in rows[:max_rows]:
        values = []
        for key in keys:
            value = row[key]
            if isinstance(value, float):
                values.append(f'{value:.6g}')
            else:
                values.append(str(value))
        print(' | '.join(values))
    if len(rows) > max_rows:
        print(f'... {len(rows) - max_rows} rows omitted')


def save_grid_png(samples_denorm, out_path, title, channel=0, cmap='viridis'):
    n = samples_denorm.shape[0]
    cols = int(np.ceil(np.sqrt(n)))
    rows = int(np.ceil(n / cols))
    imgs = samples_denorm[:, channel]
    vmin = float(np.nanmin(imgs))
    vmax = float(np.nanmax(imgs))
    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.2 * rows))
    axes = np.atleast_1d(axes).flatten()
    for idx, ax in enumerate(axes):
        ax.axis('off')
        if idx < n:
            ax.imshow(imgs[idx], cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(str(idx), fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close(fig)


def extract_features(
    samples_denorm,
    valid_mask,
    rng,
    sample_size=VALID_PIXEL_SAMPLE_SIZE,
    n_proj=N_RANDOM_PROJECTIONS,
    pixel_idx=None,
    projection_matrix=None,
):
    samples = np.asarray(samples_denorm, dtype=np.float32)
    mask = valid_mask.astype(bool).reshape(-1)
    n, c, h, w = samples.shape
    flat = samples.reshape(n, c, -1)[:, :, mask]

    features = []
    names = []
    for ch in FEATURE_CHANNELS:
        vals = flat[:, ch, :]
        channel_features = [
            vals.mean(axis=1),
            vals.std(axis=1),
            np.quantile(vals, 0.05, axis=1),
            np.quantile(vals, 0.50, axis=1),
            np.quantile(vals, 0.95, axis=1),
            vals.min(axis=1),
            vals.max(axis=1),
        ]
        channel_names = [
            f'ch{ch}_mean',
            f'ch{ch}_std',
            f'ch{ch}_q05',
            f'ch{ch}_q50',
            f'ch{ch}_q95',
            f'ch{ch}_min',
            f'ch{ch}_max',
        ]
        features.extend(channel_features)
        names.extend(channel_names)

    if pixel_idx is None:
        take = min(sample_size, flat.shape[-1])
        pixel_idx = rng.choice(flat.shape[-1], size=take, replace=False)
    flat_sub = flat[:, :, pixel_idx].reshape(n, -1).astype(np.float32, copy=False)
    if projection_matrix is None:
        projection_matrix = rng.normal(size=(flat_sub.shape[1], n_proj)).astype(np.float32) / np.sqrt(flat_sub.shape[1])
    projected = flat_sub @ projection_matrix
    for j in range(projected.shape[1]):
        features.append(projected[:, j])
        names.append(f'random_projection_{j:03d}')

    X = np.stack(features, axis=1).astype(np.float64)
    return X, names, pixel_idx, projection_matrix


def standardize_train_test(X, Y):
    pooled = np.vstack([X, Y])
    mu = pooled.mean(axis=0, keepdims=True)
    sigma = pooled.std(axis=0, keepdims=True)
    sigma = np.where(sigma < 1e-12, 1.0, sigma)
    return (X - mu) / sigma, (Y - mu) / sigma


def pairwise_sq_dists(A, B):
    A2 = np.sum(A * A, axis=1, keepdims=True)
    B2 = np.sum(B * B, axis=1, keepdims=True).T
    return np.maximum(A2 + B2 - 2 * A @ B.T, 0.0)


def energy_statistic(X, Y):
    dxy = np.sqrt(pairwise_sq_dists(X, Y))
    dxx = np.sqrt(pairwise_sq_dists(X, X))
    dyy = np.sqrt(pairwise_sq_dists(Y, Y))
    return float(2.0 * dxy.mean() - dxx.mean() - dyy.mean())


def mmd_rbf_statistic(X, Y, gamma=None):
    Z = np.vstack([X, Y])
    if gamma is None:
        d2 = pairwise_sq_dists(Z, Z)
        upper = d2[np.triu_indices_from(d2, k=1)]
        median = np.median(upper[upper > 0]) if np.any(upper > 0) else 1.0
        gamma = 1.0 / (2.0 * median)
    Kxx = np.exp(-gamma * pairwise_sq_dists(X, X))
    Kyy = np.exp(-gamma * pairwise_sq_dists(Y, Y))
    Kxy = np.exp(-gamma * pairwise_sq_dists(X, Y))
    return float(Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean()), float(gamma)


def permutation_test(X, Y, statistic_fn, n_permutations=TEST_N_PERMUTATIONS, seed=TEST_SEED):
    rng = np.random.default_rng(seed)
    Z = np.vstack([X, Y])
    n_x = X.shape[0]
    observed = statistic_fn(X, Y)
    if isinstance(observed, tuple):
        observed_value = observed[0]
        extra = observed[1:]
    else:
        observed_value = observed
        extra = ()

    perm_values = np.empty(n_permutations, dtype=np.float64)
    for p in range(n_permutations):
        perm = rng.permutation(Z.shape[0])
        Xp = Z[perm[:n_x]]
        Yp = Z[perm[n_x:]]
        value = statistic_fn(Xp, Yp)
        perm_values[p] = value[0] if isinstance(value, tuple) else value
    p_value = (1.0 + np.sum(perm_values >= observed_value)) / (n_permutations + 1.0)
    return observed_value, float(p_value), perm_values, extra


In [ ]:
transform = partial(channel_normalize, channel_mean=CHANNEL_MEAN, channel_std=CHANNEL_STD)
reference_dir = os.path.join(DATA_ROOT, REFERENCE_SPLIT)
neighbor_dir = os.path.join(DATA_ROOT, NEIGHBOR_SPLIT)

reference_dataset = NpyImageDataset(
    folder=reference_dir,
    transform=transform,
    preload=False,
    mmap_mode='r',
)
neighbor_dataset = NpyImageDataset(
    folder=neighbor_dir,
    transform=transform,
    preload=False,
    mmap_mode='r',
)

reference_norm = reference_dataset[REFERENCE_INDEX]
reference_name = reference_dataset.files[REFERENCE_INDEX]
track_mask_np = make_track_mask()
track_mask = torch.from_numpy(track_mask_np).unsqueeze(0).to(dtype=reference_norm.dtype)
observed_norm = reference_norm * track_mask

print('reference_name      =', reference_name)
print('reference_shape     =', tuple(reference_norm.shape))
print('neighbor_pool_size  =', len(neighbor_dataset))
print('track pixels        =', int(track_mask_np.sum()))
print('track coverage      =', float(track_mask_np.mean()))

make_plot(reference_norm.unsqueeze(0), CHANNEL_MEAN, CHANNEL_STD, 1, title='Reference x_ref - concentration')
make_plot(observed_norm.unsqueeze(0), CHANNEL_MEAN, CHANNEL_STD, 1, title='Condition y = x_ref * mask - concentration')
make_scalar_plot_grid(torch.from_numpy(track_mask_np).unsqueeze(0), 1, title='Track mask', cmap='gray')


In [ ]:
distances = []
for idx in tqdm(range(len(neighbor_dataset)), desc='kNN over neighbor split'):
    if EXCLUDE_REFERENCE_FROM_NEIGHBORS and neighbor_dataset.files[idx] == reference_name:
        continue
    candidate = neighbor_dataset[idx]
    distance = masked_condition_mse(candidate, observed_norm, track_mask_np)
    distances.append((distance, idx, neighbor_dataset.files[idx]))

distances = sorted(distances, key=lambda item: item[0])
neighbor_rows = []
selected_neighbor_indices = []
selected_neighbor_files = []
for rank, (distance, idx, name) in enumerate(distances[:K_NEIGHBORS]):
    neighbor_rows.append({'rank': rank, 'dataset_idx': idx, 'file': name, 'condition_mse_norm': distance})
    selected_neighbor_indices.append(idx)
    selected_neighbor_files.append(name)

neighbor_norm = torch.stack([neighbor_dataset[idx] for idx in selected_neighbor_indices])
reference_repeated = reference_norm.unsqueeze(0).repeat(K_NEIGHBORS, 1, 1, 1)
neighbor_mse_rows = physical_mse_rows(neighbor_norm, reference_norm, VALID_MASK)
for row, mse_row in zip(neighbor_rows, neighbor_mse_rows):
    row.update({key: value for key, value in mse_row.items() if key != 'idx'})

print_rows(neighbor_rows, max_rows=K_NEIGHBORS)

make_plot(neighbor_norm, CHANNEL_MEAN, CHANNEL_STD, K_NEIGHBORS, title='kNN real neighbors - concentration')
make_difference_plot(
    neighbor_norm,
    reference_repeated,
    CHANNEL_MEAN,
    CHANNEL_STD,
    K_NEIGHBORS,
    land_mask=VALID_MASK,
    title='kNN neighbors minus reference',
)

os.makedirs(CACHE_DIR, exist_ok=True)
neighbors_denorm = denormalize_tensor(neighbor_norm).numpy().astype(np.float32)
reference_denorm = denormalize_tensor(reference_norm.unsqueeze(0)).numpy()[0].astype(np.float32)
save_grid_png(neighbors_denorm, os.path.join(CACHE_DIR, 'knn_neighbors_channel0.png'), 'kNN real neighbors - channel 0')
save_grid_png(
    neighbors_denorm - reference_denorm[None],
    os.path.join(CACHE_DIR, 'knn_neighbor_differences_channel0.png'),
    'kNN real neighbors minus reference - channel 0',
    cmap='coolwarm',
)
np.save(os.path.join(CACHE_DIR, 'track_mask.npy'), track_mask_np.astype(np.float32))
np.save(os.path.join(CACHE_DIR, 'reference_denorm.npy'), reference_denorm)
print('saved neighbor preview to', os.path.join(CACHE_DIR, 'knn_neighbors_channel0.png'))
print('saved neighbor differences to', os.path.join(CACHE_DIR, 'knn_neighbor_differences_channel0.png'))


In [ ]:
def cached_sample_paths(cache_dir):
    return [os.path.join(cache_dir, f'sample_{idx:04d}.npy') for idx in range(N_MODEL_SAMPLES)]


def cache_is_compatible(cache_dir):
    meta_path = os.path.join(cache_dir, 'meta.json')
    if not os.path.exists(meta_path):
        return False
    with open(meta_path) as f:
        meta = json.load(f)
    expected = {
        'reference_split': REFERENCE_SPLIT,
        'reference_index': REFERENCE_INDEX,
        'reference_name': reference_name,
        'n_model_samples': N_MODEL_SAMPLES,
        'generation_seed': GENERATION_SEED,
        'num_timesteps': NUM_TIMESTEPS,
        'method': METHOD,
        'checkpoint_name': CHECKPOINT_NAME,
        'track_seed': TRACK_SEED,
        'n_tracks_range': list(N_TRACKS_RANGE),
    }
    return all(meta.get(key) == value for key, value in expected.items())


os.makedirs(CACHE_DIR, exist_ok=True)
sample_paths = cached_sample_paths(CACHE_DIR)
all_samples_exist = all(os.path.exists(path) for path in sample_paths)

if cache_is_compatible(CACHE_DIR) and all_samples_exist:
    print('using cached generated samples from', CACHE_DIR)
else:
    concat_run_dir = resolve_concat_run_dir()
    print('generating samples into', CACHE_DIR)
    print('concat_run_dir =', concat_run_dir)
    sampler = load_concat_sampler(concat_run_dir, CHECKPOINT_NAME, DEVICE)

    mask_batch_base = track_mask.unsqueeze(0).to(device=DEVICE)
    observed_base = observed_norm.unsqueeze(0).to(device=DEVICE)

    generated_count = 0
    for batch_start in tqdm(range(0, N_MODEL_SAMPLES, GENERATION_BATCH_SIZE), desc='concat samples'):
        batch_end = min(batch_start + GENERATION_BATCH_SIZE, N_MODEL_SAMPLES)
        batch_size = batch_end - batch_start
        seed = GENERATION_SEED + batch_start
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        mask_batch = mask_batch_base.expand(batch_size, -1, -1, -1)
        observed_batch = observed_base.expand(batch_size, -1, -1, -1)
        samples_norm = sampler.sample_conditioned(
            mask=mask_batch,
            observed=observed_batch,
            size=IMAGE_SIZE,
            num_timesteps=NUM_TIMESTEPS,
            device=DEVICE,
            method=METHOD,
        )
        samples_denorm = denormalize_tensor(samples_norm).numpy().astype(np.float32)

        for local_idx, sample in enumerate(samples_denorm):
            sample_idx = batch_start + local_idx
            np.save(sample_paths[sample_idx], sample)
            generated_count += 1

    meta = {
        'cache_dir': os.path.abspath(CACHE_DIR),
        'reference_split': REFERENCE_SPLIT,
        'neighbor_split': NEIGHBOR_SPLIT,
        'reference_index': REFERENCE_INDEX,
        'reference_name': reference_name,
        'n_model_samples': N_MODEL_SAMPLES,
        'generation_seed': GENERATION_SEED,
        'num_timesteps': NUM_TIMESTEPS,
        'method': METHOD,
        'checkpoint_name': CHECKPOINT_NAME,
        'concat_run_dir': os.path.abspath(concat_run_dir),
        'track_seed': TRACK_SEED,
        'n_tracks_range': list(N_TRACKS_RANGE),
        'k_neighbors': K_NEIGHBORS,
        'selected_neighbor_files': selected_neighbor_files,
    }
    with open(os.path.join(CACHE_DIR, 'meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    print('generated_count =', generated_count)

generated_denorm = np.stack([to_channel_first(np.load(path, mmap_mode='r')) for path in sample_paths], axis=0).astype(np.float32)
generated_norm = torch.stack([normalize_np(sample) for sample in generated_denorm])
print('generated_denorm shape =', generated_denorm.shape)

save_grid_png(generated_denorm, os.path.join(CACHE_DIR, 'generated_samples_channel0.png'), 'Generated concat samples - channel 0')
save_grid_png(
    generated_denorm - reference_denorm[None],
    os.path.join(CACHE_DIR, 'generated_differences_channel0.png'),
    'Generated samples minus reference - channel 0',
    cmap='coolwarm',
)
make_plot(generated_norm, CHANNEL_MEAN, CHANNEL_STD, min(N_MODEL_SAMPLES, 64), title='Generated concat samples - concentration')
make_difference_plot(
    generated_norm[:min(N_MODEL_SAMPLES, 64)],
    reference_norm.unsqueeze(0).repeat(min(N_MODEL_SAMPLES, 64), 1, 1, 1),
    CHANNEL_MEAN,
    CHANNEL_STD,
    min(N_MODEL_SAMPLES, 64),
    land_mask=VALID_MASK,
    title='Generated samples minus reference',
)
print('saved generated preview to', os.path.join(CACHE_DIR, 'generated_samples_channel0.png'))
print('saved generated differences to', os.path.join(CACHE_DIR, 'generated_differences_channel0.png'))


In [ ]:
generated_mse_rows = physical_mse_rows(generated_norm, reference_norm, VALID_MASK)
print('Generated sample MSE to reference:')
print_rows(generated_mse_rows, max_rows=N_MODEL_SAMPLES)

real_denorm = neighbors_denorm
fake_denorm = generated_denorm

rng = np.random.default_rng(TEST_SEED)
X_real, feature_names, pixel_idx, projection_matrix = extract_features(real_denorm, VALID_MASK, rng)
X_fake, _, _, _ = extract_features(
    fake_denorm,
    VALID_MASK,
    rng,
    pixel_idx=pixel_idx,
    projection_matrix=projection_matrix,
)
X_real_z, X_fake_z = standardize_train_test(X_real, X_fake)

energy_value, energy_p, energy_null, _ = permutation_test(
    X_real_z,
    X_fake_z,
    energy_statistic,
    n_permutations=TEST_N_PERMUTATIONS,
    seed=TEST_SEED,
)
mmd_value, mmd_p, mmd_null, mmd_extra = permutation_test(
    X_real_z,
    X_fake_z,
    mmd_rbf_statistic,
    n_permutations=TEST_N_PERMUTATIONS,
    seed=TEST_SEED + 1,
)

print('Multivariate permutation tests on standardized domain/random-projection features')
print('feature_dim      =', X_real_z.shape[1])
print('n_real           =', X_real_z.shape[0])
print('n_generated      =', X_fake_z.shape[0])
print('energy_statistic =', energy_value)
print('energy_p_value   =', energy_p)
print('mmd_rbf_stat     =', mmd_value)
print('mmd_rbf_p_value  =', mmd_p)
print('mmd_gamma        =', mmd_extra[0] if mmd_extra else None)

summary_rows = []
for col, name in enumerate(feature_names):
    if name.startswith('random_projection_'):
        continue
    real_values = X_real[:, col]
    fake_values = X_fake[:, col]
    ks = stats.ks_2samp(real_values, fake_values, alternative='two-sided', method='auto')
    summary_rows.append({
        'feature': name,
        'real_mean': float(real_values.mean()),
        'fake_mean': float(fake_values.mean()),
        'mean_diff': float(fake_values.mean() - real_values.mean()),
        'real_std': float(real_values.std(ddof=0)),
        'fake_std': float(fake_values.std(ddof=0)),
        'ks_stat': float(ks.statistic),
        'ks_p': float(ks.pvalue),
    })

print('Univariate KS tests for interpretable features:')
print_rows(summary_rows, max_rows=len(summary_rows))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(energy_null, bins=40, alpha=0.8, color='tab:blue')
axes[0].axvline(energy_value, color='black', linestyle='--', label='observed')
axes[0].set_title(f'Energy null, p={energy_p:.3g}')
axes[0].legend()
axes[1].hist(mmd_null, bins=40, alpha=0.8, color='tab:orange')
axes[1].axvline(mmd_value, color='black', linestyle='--', label='observed')
axes[1].set_title(f'MMD null, p={mmd_p:.3g}')
axes[1].legend()
plt.tight_layout()
fig.savefig(os.path.join(CACHE_DIR, 'statistical_test_nulls.png'), dpi=120, bbox_inches='tight')

with open(os.path.join(CACHE_DIR, 'statistical_tests.json'), 'w') as f:
    json.dump({
        'energy_statistic': energy_value,
        'energy_p_value': energy_p,
        'mmd_rbf_statistic': mmd_value,
        'mmd_rbf_p_value': mmd_p,
        'mmd_gamma': mmd_extra[0] if mmd_extra else None,
        'feature_dim': int(X_real_z.shape[1]),
        'n_real': int(X_real_z.shape[0]),
        'n_generated': int(X_fake_z.shape[0]),
        'univariate_features': summary_rows,
    }, f, indent=2)
print('saved test outputs to', CACHE_DIR)
